In [21]:
from dotenv import load_dotenv
load_dotenv()


False

In [12]:
from datetime import datetime, timedelta
import json
import os
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import AIMessage, HumanMessage
from openai import OpenAI

# 修正点1：从环境变量获取API密钥
client = OpenAI(
    api_key=os.getenv("sk-c446a31711e24b53bcdd9029b78e31e1"),  # 替换为有效API密钥
    base_url="https://api.deepseek.com"
)

class ChatState(TypedDict):
    messages: Annotated[List[dict], lambda x, y: x + y]
    timestamps: Annotated[List[str], lambda x, y: x + y]
    thoughts: Annotated[List[str], lambda x, y: x + y]

graph_builder = StateGraph(ChatState)

TODAY = datetime.now().strftime("%Y%m%d")
CHAT_FILE = f"chat_{TODAY}.log"

SYSTEM_PROMPT = """当前时间：{time}
作为{role}，在回复时需要：
1. 生成200字内的自然对话
2. 用【心理活动】标签记录思考过程
3. 重要信息用<important>标记"""

def ai_node(state: ChatState, role: str):
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    prompt = SYSTEM_PROMPT.format(time=current_time, role=role)
    
    history = "\n".join(
        f"[{ts}] {msg['content']}" 
        for ts, msg in zip(state["timestamps"], state["messages"])
    )
    
    try:
        response = client.chat.completions.create(
            model="deepseek-reasoner",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"历史对话：\n{history}\n请生成{role}的回复："}
            ]
        )
    except Exception as e:
        # 修正点2：添加API错误处理
        print(f"API调用失败: {str(e)}")
        return {
            "messages": [{"role": role, "content": "系统服务暂不可用"}],
            "timestamps": [current_time],
            "thoughts": ["API连接异常"]
        }
    
    content, thought = parse_response(response.choices[0].message.content)
    
    # 修正点3：增强类型校验
    if not isinstance(content, str) or not isinstance(thought, str):
        content = "响应解析异常"
        thought = "类型校验失败"
    
    return {
        "messages": [{"role": role, "content": content}],
        "timestamps": [current_time],
        "thoughts": [thought]
    }

graph_builder.add_node("AI_1", lambda state: ai_node(state, "AI助手"))
graph_builder.add_node("AI_2", lambda state: ai_node(state, "AI用户"))

graph_builder.add_edge("AI_1", "AI_2")
graph_builder.add_edge("AI_2", "AI_1")
graph_builder.set_entry_point("AI_1")

graph = graph_builder.compile()

def parse_response(text: str):
    content_part = text.split("【心理活动】")[0].strip()
    thought_part = text.split("【心理活动】")[1].strip() if "【心理活动】" in text else ""
    return str(content_part), str(thought_part)

def save_chat(state: ChatState):
    with open(CHAT_FILE, "a", encoding='utf-8') as f:
        for msg, ts, thought in zip(state["messages"], state["timestamps"], state["thoughts"]):
            if isinstance(msg, dict) and 'role' in msg and 'content' in msg:
                record = {
                    "timestamp": ts,
                    "role": msg["role"],
                    "content": msg["content"],
                    "thought": thought,
                    "is_important": "<important>" in msg["content"]
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                print(f"[{ts}] {msg['role']}: {msg['content']}")

if __name__ == "__main__":
    state = {"messages": [], "timestamps": [], "thoughts": []}
    for _ in range(3):
        try:
            state = graph.invoke(state)
            save_chat(state)
        except Exception as e:
            print(f"对话流程异常: {str(e)}")
            break

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable